# Gate: transferable scale, depth shrinkage, eligibility

`scripts/run_completion_benchmark.py` produced every number here. This notebook
only reads what that script wrote.

Three defects in the transferred prior, found on 19 August 2026.

1. `target_scale` used the singles interquartile range. Most single mutations are
   neutral, so the middle half of a DMS distribution sits near zero. One landscape
   normalised to zero and another to 0.017, and those inflated the pooled
   `order_scale`. Standard deviation replaces it.
2. The depth residual transferred at full strength where two landscapes or fewer
   reached that depth. A minimum-landscape guard now zeroes those bins.
3. `CR6261_H9_binding` reports all eleven single mutants at exactly the wild-type
   value, so a singles-only fit is undefined there. It is skipped and recorded.

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parents[1] / "src"))
from mutascape.benchmarks.reporting import (
    coverage_table,
    load_runs,
    median_table,
    paired_comparison,
    versus_baseline,
)

pd.set_option("display.width", 160)
ROOT = Path.cwd().parents[1] / "notebooks" / "iclr"


In [2]:
runs = load_runs({"gate": ROOT / "output_260819_gate"})
summary = runs["gate"]["summary"]
print("landscapes", summary["n_landscapes"], " proteins", summary["n_proteins"])
print("skipped:", [s["dataset"] for s in summary["skipped_landscapes"]])

landscapes 29  proteins 14
skipped: ['CR6261_H9_binding']


## Depth shrinkage weights

The weight is the share of the pooled depth shift that survives transfer.

In [3]:
prior = summary["final_prior"]
depth = pd.DataFrame({
    "location": pd.Series(prior["depth_location"], dtype=float),
    "between": pd.Series(prior["depth_between"], dtype=float),
    "shrinkage": pd.Series(prior["depth_shrinkage"], dtype=float),
})
depth.index = depth.index.astype(int)
depth.sort_index().round(4)

,location,between,shrinkage
2,0.0005,1.9157,0.0043
3,0.0029,3.3740,0.0090
4,0.3579,1.6673,0.3159
5,0.5480,3.6237,0.2571
6,1.2114,4.6315,0.3536
7,1.3717,7.1186,0.2967
8,1.4635,8.2400,0.2828
9,0.2070,10.0769,0.0731
10,-1.1680,5.5905,0.3110
11,-1.2951,6.2597,0.3092


## Posterior against additive ridge

`wins` counts landscapes where the posterior beats the baseline.

In [4]:
versus_baseline(runs["gate"]).round(3)

,ridge_additive,posterior,wins,n
metric,,,,
spearman,0.238,0.247,20,29
ndcg,0.889,0.925,20,29
mae,0.256,0.854,6,29
rmse,0.319,0.982,7,29
wasserstein,0.254,0.791,6,29


## What the gate did and did not fix

Rank is fixed. Spearman rises above additive ridge and NDCG follows.

Error does not move. That points at something other than the depth term, and
`260819_e1_global_epistasis.ipynb` identifies it as assay saturation.

In [5]:
coverage_table(runs).round(3)

gate           
        coverage mean_width
nominal                    
0.50       0.150      0.550
0.80       0.335      1.080
0.95       0.620      1.741